## From Building Blocks to U-Net - Understanding Architectural Innovation

### Part 1: Understanding U-Net Architecture (15 points)

U-Net was introduced by Ronneberger et al. (2015) for biomedical image segmentation. The architecture won the ISBI cell tracking challenge 2015 by a large margin.

![U-Net Architecture](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png)
*Figure 1: Original U-Net architecture (Ronneberger et al., 2015)*

### Starter Code: Basic U-Net Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

**Q1.1 (5 points):** Identify and explain the three major architectural components/concepts from our previous lectures that U-Net combines. For each component:
- Name the concept
- Explain how it's implemented in U-Net
- Describe what problem it solves

**Name of Concept 1**: Encoder

**Description of how Concept 1 is implemented in U-Net**: 

An encoder repeatedly detects patterns and generalizes an input until its key feature are compressed down into a smaller context. In this case, the image is encoded in the "Down" step by repeatendly applying 2d double convolutions followed by max pooling with a kernel size of 2. The convolutions detect patterns and the max pooling compresses the data from there. 

**Problem Concept 1 solves**: 

Concept one extracts the high level information within the image, boiling it down to key features and encoding them in a more efficient way. This also means that each step "Down" sees more of the image and is able to extract more and more high level features.

-----

**Name of Concept 2**: Decoder

**Description of how Concept 2 is implemented in U-Net**: 

Decoding is the process of taking encoded information and extrapolating it back to its original form, transfomred in some way. This step expands the information compressed into the encoder into the final form that we wish to have it in. In this case, the decoder is the "Up" step. Itrestores the image for us but with the key features identified by the encoder as the main artifacts int he decoded output.

**Problem Concept 2 solves**: 

Decoding/the "Up" step lets us access the information encoded by the encoding step and get granual information on the image instead of just a single classification. This is the step that takes the encoding and translates it into detailed information on the original image.

-----

**Name of Concept 3**: Skip Connections

**Description of how Concept 3 is implemented in U-Net**: 

Skip connections here are similar to the residual connections we saw when we studied ResNet, except that instead of adding residuals, they concatenate the results of each encoding and decoding step. Here, we create skip connections by concatenating the uncompressed information into the decoded value. 

**Problem Concept 3 solves**: 

Employing skip connections here lets us include the more specific, local information from the less-compressed "Down" stages to get both the big picture from total compression and the granular details that might have been lost in the course of encoding. 

I imagine this could be very useful in biomedical imaging where both the big picture and the smallest details can contain critical health information.



**Q1.2 (5 points):** Skip connections in U-Net use **concatenation** rather than **addition** (as in ResNet).
- What are the implications of this design choice?
- How does this affect the number of parameters?
- When might addition be preferred over concatenation?

**Implications of having skip connections in U-Net use concatenation**: 

Concatenating the data from the encoding steps preserves the information from the encoding step as separate rather than adding it on to the information encoded in the decoding step. As a result, the model can train on these as separate features and learn directly from them. 

**Effect of concatenation on number of parameters**: 

Because there are more resulting input channels to each step, there are also doing to be more resulting parameters/weights, making this a more expensive choice than residual connections.

**When addition might be preferred over concatenation**: 

Addition might be preferred when memory ot traiing time are concerns and we want to keep the number of paramters down.

**Q1.3 (5 points):** Critical Analysis:
- What are the potential limitations of the original U-Net architecture?
- How might U-Net struggle with very high-resolution images?
- Propose at least two modifications that could address these limitations.

**Potential limitations of the original U-Net architecture**: 

This architecture compresses images quite a bit, so if there is a lot of data contained in the image, it might either compress the image too much or fail to compress it to a level where general features can be identified.

The addition of skip connections could also lead to unnecessary noise that muddies the picture.

There are also a lot of activations so memory and time could become issues with large datasets and/or high resolution images.

**How U-Net might struggle with very high-resolution images**: 

With a high resolution images, inconsequential details might be picked up by the model with more importance than they deserve because there are only so many downsampling steps with a fixed kernel size. As a result, they may never reach an adequate receptive field to identify top-level features and place outsized importance on lower-level features.

Additionally, the extra datapoints mean that the model will take longer to train and require more activations, leading to space and time concerns. 

**Proposed modifications**: 

I would introduce some kind of regularization like weight decay or dropout to prevent over-training on details of the specific images in the trainingset.
I would also add residual blocks and make the network go deeper in downsampling to improve the receptive field on large images. The residiual blocks would allow for the depth to happen stably by preventing vanishing gradients. (It would also, however, increase the memory, so this would not be a useful modification if memory were an issue.) 

### Part 2: Architectural Variants (15 points)

#### Variant 1: U-Net++ (Nested U-Net)

U-Net++ introduces nested skip connections to reduce the semantic gap between encoder and decoder features.

In [ ]:
class UNetPlusPlus(nn.Module):

    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Define the nested structure
        # X^{0,0} -> X^{1,0} -> X^{2,0} -> X^{3,0} -> X^{4,0}
        #    |         |         |         |
        # X^{0,1} -> X^{1,1} -> X^{2,1} -> X^{3,1}
        #    |         |         |
        # X^{0,2} -> X^{1,2} -> X^{2,2}
        #    |         |
        # X^{0,3} -> X^{1,3}
        #    |
        # X^{0,4}

        nb_filter = [32, 64, 128, 256, 512]

        # Nested skip connections would be implemented here.
        # This is a simplified structure for illustration.
        pass

#### Variant 2: V-Net (Volumetric Convolutional Neural Networks)

V-Net extends U-Net to 3D volumetric data and adds residual connections within each stage.

In [ ]:
class VNetBlock(nn.Module):

    def __init__(self, in_channels, out_channels, num_convs):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv3d(in_channels if i == 0 else out_channels,
                     out_channels, kernel_size=3, padding=1)
            for i in range(num_convs)
        ])
        self.activation = nn.PReLU(out_channels)

        self.residual = nn.Conv3d(in_channels, out_channels, kernel_size=1) \
                        if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.residual(x)

        out = x
        for conv in self.convs:
            out = self.activation(conv(out))

        return out + residual

**Q3.1 (5 points):** U-Net++ Analysis:
- What problem does the nested skip connection architecture solve?
- How does it reduce the "semantic gap"?
- What is the computational cost compared to standard U-Net?

**Problem nested skip connection architecture solves**: 

This solves the problem of skip connections introducing noise to the model output -- i.e. extra detail that does not fit with the high-level ideas captured in the decoding process. 

**How nested skip connection architecture reduces semantic gap**: 

Nested skip connections reduce the semantic gap by introducing layers of connection between the upsampling and downsampling. This means that unnecessary features and details have more opportunitites to be smoothed out before the connection is made with the decoder.

**Computational cost of U-Net with nested skip connection architecture**: 

It multiplies the impact depending on the number of levels and channels. The number of nodes change the complexity from linear to quadratic with respect to the number of levels. These add even more computational cost because most of the additional blocks are at earlier layers where there the images are still at their highest resolution and therefore the computation is more expensive. These blocks also concatenate more times and each new block has to compute on all of those additional channels.

**Q3.2 (5 points):** V-Net Contributions:
- Besides 3D extension, what are the key innovations in V-Net?
- Why are residual connections particularly important for volumetric data?
- How does the Dice loss implementation differ for 3D data?

**Key innovations in V-Net**: 

VNet uses residual connections at each stage. 

**Why residual connections are important for volumetric data**: 

Residual connections improve a models ability to train deeply because they prevent vanishing gradients. Because moving from flat to 3D adds a spatial dimension the number of channels, kernel weights, and gradients increases. Activations also take more memeory because of the additional dimension. With more data, residual blocks can help by allowing us to add depth without ending up with vanishing gradients.

**How Dice loss differs for 3D data**: 

Dice loss on 3D data compares the data to not just how areas overlap but how much chunks of space overlap. 

**Q3.3 (5 points):** Comparative Analysis:
Create a comparison table with at least 5 criteria comparing:
- Standard U-Net
- U-Net++
- V-Net

Consider: parameter count, memory usage, suitable applications, training difficulty, and inference speed.

| Criterion | Standard U-Net | U-Net++ | V-Net |
| ----- | ----- | ----- | ----- |
| Parameter Count | normal | higher | highest due to kernels having an additional spatial dimension |
| Memory Usage | normal | much higher -- extra nodes at high resolutions | much higher -- 3d volumes mean many activations --> more memory|
| Suitable Applications | medical segmentation of easier to detect phenomena | medical segmentation on harder to detect phenomena | segmentation of 3D scans like CTs or MRIs |
| Training Difficulty | moderate | hard | most difficult due to memory requirements |
| Inference Speed | baseline | slower -- deeper network | slowest, most data to process |

## Resources

### Papers
- [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
- [UNet++: A Nested U-Net Architecture](https://arxiv.org/abs/1807.10165)
- [V-Net: Fully Convolutional Neural Networks for Volumetric Medical Image Segmentation](https://arxiv.org/abs/1606.04797)

### Implementations
- [PyTorch U-Net](https://github.com/milesial/Pytorch-UNet)
- [Segmentation Models PyTorch](https://github.com/qubvel/segmentation_models.pytorch)
- [MONAI (Medical Imaging)](https://monai.io/)